In [3]:
import json
import os
import statistics
import threading
import time
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional
from openai import OpenAI

import warnings
warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()
MODEL = "gpt-4o-mini"

In [22]:
# === Minimal agent loop (self-contained) ===
def run_agent_loop(
    messages,
    tools,
    tool_handlers,
    system="",
    max_iterations=20,
    on_tool_call=None,
    on_response=None,
    model=MODEL,
):

    # messages: list[dict] (N,) -- mutated in-place
    full = (
        ([{"role": "system", "content": system}] + messages)
        if system
        else list(messages)
    )

    for _ in range(max_iterations):
        resp = client.chat.completions.create(
            model=model,
            messages=full,
            tools=tools if tools else None,
            tool_choice="auto" if tools else None,
        )

        if on_response:
            on_response(resp)

        msg = resp.choices[0].message
        reason = resp.choices[0].finish_reason
        ser = {"role": "assistant", "content": msg.content}

        if msg.tool_calls:
            ser["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]

        full.append(ser)
        messages.append(ser)

        if reason != "tool_calls":
            return messages

        for tc in msg.tool_calls:
            name = tc.function.name
            inp = json.loads(tc.function.arguments)
            if on_tool_call:
                on_tool_call(name, inp)

            h = tool_handlers.get(name)
            try:
                result = h(**inp) if h else f"[ToolError] No handler: {name}"
            except Exception as exc:
                result = f"[ToolError] {type(exc).__name__}: {exc}"

            tm = {"role": "tool", "tool_call_id": tc.id, "content": str(result)}
            full.append(tm)
            messages.append(tm)
    return messages


print(f"Client ready. Model: {MODEL}")

Client ready. Model: gpt-4o-mini


# 1) Observability

An agent in production is a black box. Without observability you cannot answer basic questions:
- Why did the agent take 12 seconds?
- Which tool was called 200 times?
- Why did this session cost $3?

**Observability = Logs + Metrics + Traces.**

- **Logs**: structured JSON records of every event (message received,
  tool called, LLM response received, error occurred).
- **Metrics**: aggregated numbers (tokens/turn, latency percentiles,
  error rates, cache hit ratio).
- **Traces**: causal chains linking a user message to every downstream
  action it triggered (span tree).

| Event | What to log |
|-------|-------------|
| LLM call start | model, prompt token count, session_id |
| LLM call end | latency_ms, output tokens, finish_reason |
| Tool call | tool_name, input summary, start_time |
| Tool result | output length, duration_ms, error? |
| Compaction | before_tokens, after_tokens, layer used |
| Session start/end | session_id, channel, turn count |

1. Each user request is a root **span**.
2. Every LLM call and tool call is a child span.
3. The trace is the full tree of spans.
4. This mirrors OpenTelemetry's data model

## 1.1 Event Log

In [23]:
@dataclass
class Span:
    # One unit of work in a trace.
    span_id: str
    parent_id: Optional[str]
    name: str
    start_ms: float
    end_ms: Optional[float] = None
    attrs: dict = field(default_factory=dict)
    error: Optional[str] = None

    @property
    def duration_ms(self):
        if self.end_ms is None:
            return None
        return round(self.end_ms - self.start_ms, 2)

    def finish(self, **attrs):
        self.end_ms = time.time() * 1000
        self.attrs.update(attrs)
        return self

    def to_dict(self):
        return {**self.__dict__, "duration_ms": self.duration_ms}


class Tracer:
    """
    Collects spans into traces. One Tracer per session.
    Thread-safe (multiple tool calls can be in flight simultaneously).
    """

    def __init__(self, session_id):
        self.session_id = session_id
        # _spans: list[Span] (num_spans,)
        self._spans: list = []
        self._lock = threading.Lock()

    def start_span(self, name, parent_id=None, **attrs):
        # name: str -> Span (call .finish() when done)
        span = Span(
            span_id=uuid.uuid4().hex[:8],
            parent_id=parent_id,
            name=name,
            start_ms=time.time() * 1000,
            attrs=attrs,
        )
        with self._lock:
            self._spans.append(span)
        return span

    def record_error(self, span, error):
        span.error = str(error)
        span.finish()

    def summary(self):
        # -> str (human-readable trace summary)
        with self._lock:
            spans = list(self._spans)
        if not spans:
            return "No spans recorded."

        lines = [f"Trace [{self.session_id}]: {len(spans)} spans"]

        for s in spans:
            indent = "  " if s.parent_id else ""
            dur = f"{s.duration_ms:.0f}ms" if s.duration_ms else "running"
            err = f" ERROR: {s.error}" if s.error else ""
            lines.append(f"{indent}  {s.name} [{dur}]{err}")
        return "\n".join(lines)


class MetricsStore:
    """
    Aggregates numerical metrics with rolling window support.
    Thread-safe.
    """

    def __init__(self, window_size=100):
        # _data: dict[str, deque[float]] (num_metrics,)
        self._data: dict = defaultdict(lambda: deque(maxlen=window_size))
        self._lock = threading.Lock()

    def record(self, key, value):
        # key: str, value: float -> None
        with self._lock:
            self._data[key].append(value)

    def stats(self, key):
        # key: str -> dict (count, mean, p50, p95, p99, min, max)
        with self._lock:
            vals = list(self._data.get(key, []))

        if not vals:
            return {"count": 0}

        sorted_vals = sorted(vals)
        n = len(sorted_vals)

        def pct(p):
            return sorted_vals[min(int(n * p / 100), n - 1)]

        return {
            "count": n,
            "mean": round(statistics.mean(vals), 2),
            "p50": round(pct(50), 2),
            "p95": round(pct(95), 2),
            "p99": round(pct(99), 2),
            "min": round(min(vals), 2),
            "max": round(max(vals), 2),
        }

    def report(self):
        # -> str (all metrics)
        with self._lock:
            keys = list(self._data.keys())
        lines = []
        for k in keys:
            s = self.stats(k)
            lines.append(f"  {k}: {s}")
        return "Metrics:\n" + ("\n".join(lines) or "  (empty)")


class AgentLogger:
    """
    Structured JSON logger for agent events.
    Writes to both stdout (pretty) and an append-only JSONL file.
    """

    def __init__(self, log_path=".logs/agent.jsonl"):
        self._path = Path(log_path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self.metrics = MetricsStore()

    def _write(self, event_type, **fields):
        record = {"ts": datetime.now().isoformat(), "event": event_type, **fields}

        with open(self._path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
        return record

    def llm_call(self, session_id, model, prompt_tokens):

        return self._write(
            "llm_call", session_id=session_id, model=model, prompt_tokens=prompt_tokens
        )

    def llm_response(
        self, session_id, latency_ms, output_tokens, finish_reason, cached=False
    ):

        self.metrics.record("llm_latency_ms", latency_ms)
        self.metrics.record("output_tokens", output_tokens)
        return self._write(
            "llm_response",
            session_id=session_id,
            latency_ms=latency_ms,
            output_tokens=output_tokens,
            finish_reason=finish_reason,
            cached=cached,
        )

    def tool_call(self, session_id, tool_name, input_summary):

        self.metrics.record("tool_calls_total", 1)
        return self._write(
            "tool_call", session_id=session_id, tool=tool_name, input=input_summary
        )

    def tool_result(self, session_id, tool_name, duration_ms, output_len, error=None):

        self.metrics.record(f"tool_{tool_name}_ms", duration_ms)

        return self._write(
            "tool_result",
            session_id=session_id,
            tool=tool_name,
            duration_ms=duration_ms,
            output_len=output_len,
            error=error,
        )

    def error(self, session_id, message, exc=None):
        self.metrics.record("errors_total", 1)
        return self._write(
            "error",
            session_id=session_id,
            message=message,
            exc=str(exc) if exc else None,
        )


# === Demo ===
logger = AgentLogger()
tracer = Tracer(session_id="demo_session")

# Simulate a turn
root = tracer.start_span("user_turn", user_message="What is 2+2?")
llm = tracer.start_span("llm_call", parent_id=root.span_id, model=MODEL)
logger.llm_call("demo_session", MODEL, prompt_tokens=50)
llm.finish(output_tokens=20, finish_reason="stop")
logger.llm_response(
    "demo_session", latency_ms=50, output_tokens=20, finish_reason="stop"
)
root.finish()

print(tracer.summary())
print()
print(logger.metrics.report())

Trace [demo_session]: 2 spans
  user_turn s]
    llm_call s]

Metrics:
  llm_latency_ms: {'count': 1, 'mean': 50, 'p50': 50, 'p95': 50, 'p99': 50, 'min': 50, 'max': 50}
  output_tokens: {'count': 1, 'mean': 20, 'p50': 20, 'p95': 20, 'p99': 20, 'min': 20, 'max': 20}


# 2) Cost and Token Budgeting

LLM APIs charge per token. A runaway agent (infinite loop, accidentally reading a 100MB file, spawning too many subagents) can burn $100 in minutes.

Token budgeting adds two controls:
1. **Per-session budget**: a session cannot exceed N total tokens.
2. **Per-turn budget**: a single LLM call cannot exceed M output tokens.

Both are enforced before the API call, not after. By the time you see
the bill it is too late.


In [38]:
class BudgetExceeded(Exception):
    pass


# OpenAI list pricing per 1M tokens (USD)
COST_TABLE = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60, "cache": 0.075},
    "gpt-4o": {"input": 2.50, "output": 10.00, "cache": 1.25},
}


@dataclass
class TokenBudget:
    # Tracks token usage and enforces per-session and per-turn limits.
    session_limit: int = 100_000  # max total tokens for this session
    turn_limit: int = 4_096  # max output tokens per turn
    model: str = "gpt-4o-mini"

    # Running totals
    total_input: int = 0
    total_output: int = 0
    total_cached: int = 0
    turns: int = 0

    @property
    def total_tokens(self):
        return self.total_input + self.total_output

    @property
    def remaining(self):
        return max(0, self.session_limit - self.total_tokens)

    def consume(self, input_tokens, output_tokens, cached_tokens=0):
        """
        Record tokens from one LLM call.
        input_tokens: int, output_tokens: int -> None | raises BudgetExceeded
        """

        self.total_input += input_tokens - cached_tokens
        self.total_output += output_tokens
        self.total_cached += cached_tokens
        self.turns += 1
        if self.total_tokens > self.session_limit:
            raise BudgetExceeded(
                f"Session budget exceeded: {self.total_tokens} > {self.session_limit} tokens"
            )

    def check_turn_limit(self, requested_max_tokens):
        """
        Clamp requested_max_tokens to what the budget allows.
        requested_max_tokens: int -> int (safe max_tokens to pass to the API)
        """
        available = min(self.remaining, self.turn_limit)
        return min(requested_max_tokens, available)

    def cost_usd(self):
        # -> float (estimated USD spent so far)
        table = COST_TABLE.get(self.model, COST_TABLE["gpt-4o-mini"])
        cost = (self.total_input / 1_000_000) * table["input"]
        cost += (self.total_output / 1_000_000) * table["output"]
        cost += (self.total_cached / 1_000_000) * table["cache"]
        return round(cost, 6)

    def report(self):
        return (
            f"Budget Report [{self.model}]\n"
            f"  Turns:          {self.turns}\n"
            f"  Input tokens:   {self.total_input:,}\n"
            f"  Output tokens:  {self.total_output:,}\n"
            f"  Cached tokens:  {self.total_cached:,}\n"
            f"  Total tokens:   {self.total_tokens:,} / {self.session_limit:,}\n"
            f"  Remaining:      {self.remaining:,}\n"
            f"  Est. cost:      ${self.cost_usd():.4f} USD"
        )


def make_budget_hook(budget, logger=None):
    """
    Factory: returns an on_response hook that tracks usage and enforces budget.
    budget: TokenBudget -> Callable(response) -> None
    """

    def hook(response):
        usage = response.usage
        cached = getattr(usage, "prompt_tokens_details", None)
        cached_tokens = getattr(cached, "cached_tokens", 0) if cached else 0
        try:
            budget.consume(
                input_tokens=usage.prompt_tokens,
                output_tokens=usage.completion_tokens,
                cached_tokens=cached_tokens,
            )
        except BudgetExceeded as exc:
            if logger:
                logger.error("budget", str(exc))
            raise

    return hook


# === Demo ===
budget = TokenBudget(session_limit=10_000, turn_limit=500, model="gpt-4o")

# Simulate consuming tokens across 3 turns
budget.consume(input_tokens=300, output_tokens=150)
budget.consume(input_tokens=450, output_tokens=200, cached_tokens=200)
budget.consume(input_tokens=600, output_tokens=180)

print(budget.report())
print(f"\nTurn limit (requesting 1000): safe max = {budget.check_turn_limit(1000)}")

Budget Report [gpt-4o]
  Turns:          3
  Input tokens:   1,150
  Output tokens:  530
  Cached tokens:  200
  Total tokens:   1,680 / 10,000
  Remaining:      8,320
  Est. cost:      $0.0277 USD

Turn limit (requesting 1000): safe max = 500


# 3) Prompt Caching

OpenAI caches prompt prefixes automatically. If your system prompt is 2_000 tokens and you make 100 calls, you pay for 2_000 tokens once (full price) and 99 times at the cache rate (~50% discount).

**This only works if the prefix is identical across calls.**

Any change to the start of the prompt busts the cache. The golden rule:

```
STABLE (cache-friendly)    DYNAMIC (cache-busting)
  BASE prompt                current timestamp
  IDENTITY.md                MEMORY.md (changes per turn)
  SOUL.md                    CONTEXT (changes per turn)
  TOOLS.md
  HEARTBEAT.md
```

**Cache-Friendly Prompt Structure**

```
[STABLE PREFIX --- never changes]
  Layer 1: BASE
  Layer 2: IDENTITY
  Layer 3: SOUL
  Layer 4: TOOLS

[DYNAMIC SUFFIX --- changes every turn]
  Layer 5: MEMORY  (user-specific, changes rarely)
  Layer 6: SKILLS  (session-specific)
  Layer 7: CONTEXT (changes every turn: time, channel, peer)
  Layer 8: HEARTBEAT (only in heartbeat mode)
```

The stable prefix should be as long as possible to maximise cache hits.
Put the most stable content first, the most volatile content last.

OpenAI returns `usage.prompt_tokens_details.cached_tokens` on each
response. Tracking this tells you how well your prompt structure is
working and how much money you are saving.

In [40]:
import tiktoken


@dataclass
class CacheStats:
    # Tracks cache performance across an agent session.
    total_prompt_tokens: int = 0
    total_cached_tokens: int = 0
    calls: int = 0

    def record(self, prompt_tokens, cached_tokens):
        self.total_prompt_tokens += prompt_tokens
        self.total_cached_tokens += cached_tokens
        self.calls += 1

    @property
    def hit_rate(self):
        if self.total_prompt_tokens == 0:
            return 0.0
        return round(self.total_cached_tokens / self.total_prompt_tokens, 3)

    def savings_usd(self, model="gpt-4o-mini"):

        # Cost saved vs paying full price for cached tokens.
        table = COST_TABLE.get(model, COST_TABLE["gpt-4o-mini"])
        full_rate = table["input"]
        cache_rate = table["cache"]
        saved_per_m = full_rate - cache_rate
        return round((self.total_cached_tokens / 1_000_000) * saved_per_m, 6)

    def report(self, model="gpt-4o"):
        return (
            f"Cache Stats:\n"
            f"  API calls:           {self.calls}\n"
            f"  Total prompt tokens: {self.total_prompt_tokens:,}\n"
            f"  Cached tokens:       {self.total_cached_tokens:,}\n"
            f"  Cache hit rate:      {self.hit_rate:.1%}\n"
            f"  Savings:             ${self.savings_usd(model):.4f} USD"
        )


class CacheAwarePromptBuilder:
    """
    Builds prompts with stable prefix and dynamic suffix separated.
    The stable prefix should be maximally long for best cache performance.
    """

    def __init__(self, stable_layers, model="gpt-4o"):
        """
        stable_layers: list[str] -- content that never changes across calls
        Joined once at construction; never rebuilt.
        stable_layers: list[str] (num_stable,) -> str (cached prefix)
        """

        self._stable = "\n\n".join(stable_layers)
        self._encoder = tiktoken.encoding_for_model(model)
        self._stable_tokens = len(self._encoder.encode(self._stable))

    def build(self, dynamic_layers):
        # dynamic_layers: list[str] -- content that changes every call
        dynamic = "\n\n".join(dynamic_layers)

        # (full prompt = stable prefix + dynamic suffix)
        return self._stable + ("\n\n" + dynamic if dynamic else "")

    @property
    def stable_preview(self):
        return self._stable[:200] + ("..." if len(self._stable) > 200 else "")


def make_cache_hook(stats):
    # Factory: returns an on_response hook that records cache hit data.
    # stats: CacheStats -> Callable(response) -> None
    def hook(response):
        usage = response.usage
        cached = (
            getattr(getattr(usage, "prompt_tokens_details", None), "cached_tokens", 0)
            or 0
        )

        stats.record(prompt_tokens=usage.prompt_tokens, cached_tokens=cached)

    return hook


# === Demo ===
stable_layers = [
    "# Role\nYou are a helpful AI assistant with tool use capabilities.",
    "# Identity\nYou are Claw, deployed in a messaging gateway.",
    "# Soul\nBe concise, proactive, and honest. Avoid filler phrases.",
    "# Tools\nbash: run shell commands | file_read: read files | file_write: write files",
]

dynamic_layers = [
    f"# Context\nTime: {datetime.now().isoformat()}\nChannel: telegram\nPeer: 12345",
    "# Memory\nUser prefers bullet points.",
]

builder = CacheAwarePromptBuilder(stable_layers)
full_prompt = builder.build(dynamic_layers)

enc = tiktoken.encoding_for_model("gpt-4o-mini")
full_tokens = len(enc.encode(full_prompt))

print(f"Stable prefix: {builder._stable_tokens} tokens")
print(f"Full prompt:   {full_tokens} tokens")
print(f"Cache-able:    {builder._stable_tokens / full_tokens:.1%} of prompt")
print()

cache_stats = CacheStats()
# Simulate 10 calls with 70% cache hit rate
for i in range(10):
    cached = int(800 * 0.7) if i > 0 else 0  # first call has no cache
    cache_stats.record(prompt_tokens=1_000, cached_tokens=cached)
print(cache_stats.report())

Stable prefix: 63 tokens
Full prompt:   104 tokens
Cache-able:    60.6% of prompt

Cache Stats:
  API calls:           10
  Total prompt tokens: 10,000
  Cached tokens:       5,040
  Cache hit rate:      50.4%
  Savings:             $0.0428 USD


# 4) Multi-Model Routing

Not all tasks need a frontier model. A routing layer that sends simple
requests to a cheaper model and complex ones to a more capable model
can cut costs by 70-90% without a noticeable quality drop.

| Signal | Simple task | Complex task |
|--------|-------------|--------------|
| Message length | < 50 words | > 200 words |
| Tool calls expected | 0 | 3+ |
| Keyword heuristics | 'what time', 'hello' | 'refactor', 'debug', 'analyse' |
| Explicit user flag | `--fast` | `--deep` |
| Confidence threshold | classifier > 0.9 | classifier < 0.5 |

The simplest effective strategy is a **keyword + length heuristic**.
A more sophisticated approach calls a tiny classifier model (gpt-4o-mini)
to score complexity before routing.


In [41]:
SYSTEM_ROUTE_PROMPT = """
You are a model routing assistant. Determine if the user request is 'simple' or 'complex'. \
Simple tasks (greetings, basic facts, short queries) use 'fast'. \
Complex tasks (refactoring, debugging, deep analysis) use 'smart'. \
Respond ONLY in JSON format like: {\"route\": \"fast\" or \"smart\", \"reason\": \"<brief reason>\"}
"""


class RoutingDecision:
    def __init__(self, model, reason, confidence):
        self.model = model
        self.reason = reason
        self.confidence = confidence  # 0.0 to 1.0

    def __repr__(self):
        return f"RoutingDecision(model={self.model!r}, reason={self.reason!r}, conf={self.confidence:.2f})"


class ModelRouter:
    """
    Routes requests to fast or smart models based on LLM scoring.
    """

    def __init__(self, fast="gpt-4o-mini", smart="gpt-4o"):
        self.fast = fast
        self.smart = smart
        self._log: list = []

    def route(self, text, expected_tools=0, force=None):
        # text: str, expected_tools: int, force: str|None -> RoutingDecision
        if force:
            return RoutingDecision(force, "forced", 1.0)

        try:
            resp = client.chat.completions.create(
                model=self.fast,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_ROUTE_PROMPT,
                    },
                    {"role": "user", "content": text[:1000]},
                ],
                response_format={"type": "json_object"},
                max_completion_tokens=50,
            )
            content = resp.choices[0].message.content
            if not content:
                content = '{"route": "fast", "reason": "empty_response"}'

            result = json.loads(content)
            route_choice = result.get("route", "fast").lower()
            reason = result.get("reason", "llm_evaluated")

            if route_choice == "smart":
                d = RoutingDecision(self.smart, reason, 0.90)
            else:
                d = RoutingDecision(self.fast, reason, 0.90)

        except Exception as e:
            d = RoutingDecision(self.fast, f"fallback_due_to_error: {e}", 0.50)

        self._log.append({"text": text[:60], "model": d.model, "reason": d.reason})
        return d

    def routing_stats(self):
        # -> dict (fast_count, smart_count, fast_pct)
        fast = sum(1 for r in self._log if r["model"] == self.fast)
        smart = sum(1 for r in self._log if r["model"] == self.smart)
        total = len(self._log)
        return {
            "fast": fast,
            "smart": smart,
            "total": total,
            "fast_pct": f"{fast/total:.0%}" if total else "N/A",
        }


# === Demo ===
router = ModelRouter(fast="gpt-4o-mini", smart="gpt-4o")

test_cases = [
    "Hello there!",
    "What time is it?",
    "Refactor the authentication module to use OAuth2 with PKCE.",
    "Why does my Redis connection keep dropping under load?",
    "List the top 3 Python web frameworks.",
    "Analyse the performance bottleneck in my database query pipeline "
    "and suggest an indexing strategy to reduce p99 latency below 50ms.",
]

print("Routing decisions:")
for text in test_cases:
    d = router.route(text)
    print(f"  [{d.model:15}] {text[:55]}... -> {d.reason}")

print(f"\nStats: {router.routing_stats()}")

Routing decisions:
  [gpt-4o-mini    ] Hello there!... -> Greeting and simple interaction.
  [gpt-4o-mini    ] What time is it?... -> This is a straightforward query requesting a basic fact.
  [gpt-4o         ] Refactor the authentication module to use OAuth2 with P... -> The task involves refactoring code and requires a deep understanding of OAuth2 with PKCE.
  [gpt-4o         ] Why does my Redis connection keep dropping under load?... -> The request requires troubleshooting and analyzing connection issues under load conditions.
  [gpt-4o-mini    ] List the top 3 Python web frameworks.... -> The request is a basic fact query.
  [gpt-4o         ] Analyse the performance bottleneck in my database query... -> The request involves analyzing performance issues and suggesting an indexing strategy, which requires deep analysis and understanding of database optimization.

Stats: {'fast': 3, 'smart': 3, 'total': 6, 'fast_pct': '50%'}


# 5) Agent Testing

Testing agents is hard because they are non-deterministic: the same
prompt can produce different tool call sequences on different runs.
Three strategies that work:

1. **Mock LLM testing**: replace the LLM with a deterministic mock.
   Test that your harness correctly handles tool calls, errors, compaction.
   Fast (no API calls), reliable, cheap.

2. **Golden output testing**: record one real LLM run, save it as a
   golden fixture. Replay it to verify your harness produces the same
   final output given the same messages.

3. **Behavioural assertions**: run against the real LLM and assert on
   *properties* (response length, tool was called, no error in output)
   rather than exact text.


In [43]:
from unittest.mock import MagicMock, patch


@dataclass
class MockLLMResponse:
    # Represents one scripted LLM response for testing.
    text: Optional[str] = None
    tool_calls: list = field(default_factory=list)
    # tool_calls: list[(name: str, args: dict)]


class MockLLM:
    """
    Deterministic LLM mock for unit testing agent harness code.
    Replaces client.chat.completions.create with scripted responses.
    Does not make any API calls.
    """

    def __init__(self, responses):
        # responses: list[MockLLMResponse] -- consumed in order
        self._queue = deque(responses)
        self._calls: list = []  # all calls recorded here

    def _make_response(self, scripted):
        # Build an object that looks like a real OpenAI ChatCompletion.
        choice = MagicMock()
        choice.message.content = scripted.text
        choice.message.tool_calls = None
        choice.finish_reason = "stop"

        if scripted.tool_calls:
            choice.finish_reason = "tool_calls"
            tcs = []

            for name, args in scripted.tool_calls:
                tc = MagicMock()
                tc.id = uuid.uuid4().hex[:8]
                tc.function.name = name
                tc.function.arguments = json.dumps(args)
                tcs.append(tc)
            choice.message.tool_calls = tcs

        resp = MagicMock()
        resp.choices = [choice]
        resp.usage.prompt_tokens = 100
        resp.usage.completion_tokens = 50
        return resp

    def create(self, **kwargs):
        # Drop-in replacement for client.chat.completions.create
        self._calls.append(kwargs)
        if not self._queue:
            # Default: empty stop response when queue is exhausted
            scripted = MockLLMResponse(text="(no more scripted responses)")
        else:
            scripted = self._queue.popleft()
        return self._make_response(scripted)

    def assert_tool_called(self, tool_name):
        called = []
        for call in self._calls:
            msgs = call.get("messages", [])
            for m in msgs:
                if isinstance(m, dict) and m.get("role") == "tool":
                    called.append(m.get("content", ""))
        # Check that any assistant message requested this tool
        for call in self._calls:
            for m in call.get("messages", []):
                if isinstance(m, dict):
                    for tc in m.get("tool_calls", []) or []:
                        if (
                            isinstance(tc, dict)
                            and tc.get("function", {}).get("name") == tool_name
                        ):
                            return True
        raise AssertionError(f"Tool {tool_name!r} was never called.")

    @property
    def call_count(self):
        return len(self._calls)


class GoldenTest:
    """
    Record a real agent run; replay it to verify correctness.
    The golden fixture captures the full messages[] list.
    """

    def __init__(self, fixture_path):
        self._path = Path(fixture_path)

    def record(self, messages):
        # Save messages as the golden fixture.
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._path.write_text(json.dumps(messages, indent=2, default=str))
        print(f"Recorded {len(messages)} messages to {self._path}")

    def load(self):
        # Load the golden fixture.
        if not self._path.exists():
            raise FileNotFoundError(f"Golden fixture not found: {self._path}")
        return json.loads(self._path.read_text())

    def assert_final_text_contains(self, messages, substring):
        # Assert the last assistant message contains a substring.
        last = next(
            (
                m["content"]
                for m in reversed(messages)
                if m.get("role") == "assistant" and m.get("content")
            ),
            None,
        )
        if last is None:
            raise AssertionError("No assistant message found.")
        if substring.lower() not in last.lower():
            raise AssertionError(f"{substring!r} not found in: {last[:100]}")
        return True


# === Demo 1: Mock LLM test (Unit Testing Harness) ===
print("=== Demo 1: Mock LLM Test (Infrastructure) ===")
mock = MockLLM(
    responses=[
        MockLLMResponse(tool_calls=[("bash", {"command": "echo hello"})]),
        MockLLMResponse(text="The command output was: hello"),
    ]
)

# Patch the real client with our mock
# Intercept client wotk MockLLM, and reply `Call bash tool!`
# Test whether the loop able to find the bash tool and run the function
with patch.object(client.chat.completions, "create", side_effect=mock.create):
    messages = [{"role": "user", "content": "Run echo hello and tell me the output."}]
    run_agent_loop(
        messages=messages,
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "bash",
                    "description": "Run shell",
                    "parameters": {
                        "type": "object",
                        "properties": {"command": {"type": "string"}},
                        "required": ["command"],
                    },
                },
            }
        ],
        tool_handlers={"bash": lambda command: "hello"},
        system="You are a test assistant.",
    )

print(f"Mock LLM test passed!")
print(f"  LLM calls made: {mock.call_count}")
print(f'  Final reply: {messages[-1]["content"]}')

# Verify the bash tool was called
try:
    mock.assert_tool_called("bash")
    print("  assert_tool_called(bash): PASS")
except AssertionError as e:
    print(f"  FAIL: {e}")


# === Demo 2: Real LLM test (Integration Testing Intelligence) ===
print("\n=== Demo 2: Real LLM Test (Intelligence) ===")
real_messages = [
    {"role": "user", "content": "Calculate 15 * 7 and reply with just the number."}
]

# This runs against the REAL API, proving end-to-end capabilities
run_agent_loop(
    messages=real_messages,
    tools=[],
    tool_handlers={},
    system="You are a helpful assistant.",
)
print(f"Real LLM test passed!")
print(f'  Final reply: {real_messages[-1].get("content")}')

=== Demo 1: Mock LLM Test (Infrastructure) ===
Mock LLM test passed!
  LLM calls made: 2
  Final reply: The command output was: hello
  assert_tool_called(bash): PASS

=== Demo 2: Real LLM Test (Intelligence) ===
Real LLM test passed!
  Final reply: 105


# 6) Complete ProductionAgent

A single `run()` call that wires the ops layer together: model routed,
agent loop run with tracing and logging hooks, token budget enforced,
cache hit rate tracked.


In [45]:
class ProductionAgent:
    """
    Wraps the agent loop with the ops layer: routing, tracing, budget, cache stats.
    Designed to be instantiated once per deployment.
    """

    def __init__(self, session_token_limit=100_000, turn_token_limit=4_096):
        self.logger = AgentLogger()
        self.router = ModelRouter(fast="gpt-4o-mini", smart="gpt-4o")
        self.budget = TokenBudget(
            session_limit=session_token_limit,
            turn_limit=turn_token_limit,
            model=self.router.smart,
        )
        self.cache_stats = CacheStats()

        # _sessions: dict[str, list] -- per-peer conversation history
        self._sessions: dict = {}

    def run(self, text, peer_id="anonymous", tools=None, tool_handlers=None):
        # text: str, peer_id: str -> str (agent reply) | error message
        session_id = f"{peer_id}_{datetime.now().date()}"
        model = self.router.route(text).model

        tracer = Tracer(session_id=session_id)
        root = tracer.start_span("user_turn", peer_id=peer_id, model=model)

        budget_hook = make_budget_hook(self.budget, self.logger)
        cache_hook = make_cache_hook(self.cache_stats)

        def on_response(resp):
            budget_hook(resp)
            cache_hook(resp)
            self.logger.llm_response(
                session_id=session_id,
                latency_ms=0,
                output_tokens=resp.usage.completion_tokens,
                finish_reason=resp.choices[0].finish_reason,
            )

        messages = list(self._sessions.get(peer_id, []))
        messages.append({"role": "user", "content": text})

        try:
            run_agent_loop(
                messages=messages,
                tools=tools or [],
                tool_handlers=tool_handlers or {},
                system=f"You are a helpful AI agent. Session: {session_id}.",
                on_response=on_response,
                model=model,
            )
        except BudgetExceeded as exc:
            root.finish(error=str(exc))
            return f"[BudgetExceeded] {exc}"
        except Exception as exc:
            root.finish(error=str(exc))
            self.logger.error(session_id, str(exc), exc=exc)
            return f"[Error] {exc}"

        root.finish()
        self._sessions[peer_id] = messages
        return next(
            (
                m["content"]
                for m in reversed(messages)
                if m["role"] == "assistant" and m["content"]
            ),
            "(no reply)",
        )

    def ops_report(self):
        return (
            f"=== Ops Report ===\n"
            f"{self.budget.report()}\n\n"
            f"{self.cache_stats.report()}\n\n"
            f"{self.logger.metrics.report()}\n\n"
            f"Model routing: {self.router.routing_stats()}"
        )


# Capstone Demo
print("Initialising ProductionAgent...")
agent = ProductionAgent()

print("=" * 60)
print("Turn 1: Simple greeting")
print("=" * 60)
r1 = agent.run("Hello! What can you do?", peer_id="alice")
print(f"Reply: {r1[:200]}")

print("\n" + "=" * 60)
print("Turn 2: Complex question (multi-turn context retained)")
print("=" * 60)
r2 = agent.run(
    "Explain how prompt caching changes the cost of a long agent session, in 2 sentences.",
    peer_id="alice",
)
print(f"Reply: {r2[:200]}")

print("\n" + "=" * 60)
print("Ops Report")
print("=" * 60)
print(agent.ops_report())


Initialising ProductionAgent...
Turn 1: Simple greeting
Reply: Hello! I can assist you with a variety of tasks, such as answering questions, providing information on various topics, helping with problem-solving, generating ideas, and much more. If you have someth

Turn 2: Complex question (multi-turn context retained)
Reply: The circuit breaker pattern is a design pattern used in software development to enhance the stability of a system by preventing it from repeatedly trying to execute an operation that is likely to fail

Ops Report
=== Ops Report ===
Budget Report [gpt-4o]
  Turns:          2
  Input tokens:   138
  Output tokens:  121
  Cached tokens:  0
  Total tokens:   259 / 100,000
  Remaining:      99,741
  Est. cost:      $0.0050 USD

Cache Stats:
  API calls:           2
  Total prompt tokens: 138
  Cached tokens:       0
  Cache hit rate:      0.0%
  Savings:             $0.0000 USD

Metrics:
  llm_latency_ms: {'count': 2, 'mean': 0, 'p50': 0, 'p95': 0, 'p99': 0, 'min': 0, '

# Summary

```
+--------------------------------------------------------+
|                    Core                                |
|  The engine: how an agent thinks and acts              |
|                                                        |
|  run_agent_loop    TodoManager    FileTaskStore        |
|  SkillLibrary      compact_messages                    |
|  AutonomousWorker  WorktreeManager  AgentHarness       |
+--------------------------|-----------------------------+
                           |
+--------------------------|-----------------------------+
|                    Operation                           |
|  The factory: how you run it reliably at scale         |
|                                                        |
|  AgentLogger   TokenBudget   CacheStats                |
|  ModelRouter   MockLLM       GoldenTest                |
|  ProductionAgent                                       |
+--------------------------------------------------------+
```

## Further Reading

- [shareAI-lab/learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) -- agent core reference (12 sessions)
- [OpenTelemetry Python](https://opentelemetry-python.readthedocs.io/) -- production tracing
- [openai-cookbook](https://github.com/openai/openai-cookbook) -- prompt caching patterns